# 02. Data Validation & Cleaning

## Objective
Apply reproducible schema validation and data cleaning rules from `src/data/preprocessing.py` while keeping `data/raw/diabetes_data.csv` completely unchanged.

Key best practices enforced here:
- Validate schema, duplicates, missing values, and clinical ordering checks before transformations.
- Remove non-informative columns (`PatientID`, constant `DoctorInCharge`).
- Flag physiological anomalies (`DiastolicBP >= SystolicBP`) transparently rather than silently dropping records.
- Split into training and test sets **before** fitting any scaling or encoding transformer.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.preprocessing import (
    load_raw_data,
    validate_dataset,
    clean_dataset,
    create_stratified_splits,
    build_preprocessor,
)

raw_df = load_raw_data()
report = validate_dataset(raw_df)
report

### Analysis: Validation Report
The validation check confirms **1,879 rows**, **46 columns**, **0 missing cells**, **0 exact duplicates**, 1 constant column (`DoctorInCharge`), and **158 records** where `DiastolicBP >= SystolicBP`. Next, we apply `clean_dataset()` and inspect the resulting columns.

In [ ]:
cleaned_df = clean_dataset(raw_df, drop_invalid_bp=False)
print(f"Cleaned shape: {cleaned_df.shape}")
print(f"IsInvalidBP counts:\n{cleaned_df['IsInvalidBP'].value_counts()}")

processed_path = PROJECT_ROOT / "data" / "processed" / "diabetes_cleaned.csv"
cleaned_df.to_csv(processed_path, index=False)
print(f"Saved cleaned dataset to: {processed_path.relative_to(PROJECT_ROOT)}")

### Analysis: Leakage-Safe Train/Test Split
Keeping `IsInvalidBP` as an explicit indicator retains all 1,879 patient records for exploratory analysis while letting downstream sensitivity checks test excluding flagged BP records. Below, we verify that `create_stratified_splits()` preserves the ~60/40 `Diagnosis` class ratio before fitting `build_preprocessor()` solely on `X_train`.

In [ ]:
X_train, X_test, y_train, y_test = create_stratified_splits(cleaned_df, test_size=0.2, random_state=42)
preprocessor = build_preprocessor(X_train.columns)

X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

print(f"X_train transformed shape: {X_train_t.shape}, positive rate: {y_train.mean():.4f}")
print(f"X_test transformed shape:  {X_test_t.shape}, positive rate: {y_test.mean():.4f}")

### Summary
Both train and test splits maintain the exact class proportion (`~40.02%` positive `Diagnosis`), and preprocessing parameters (median imputers, scalers, encoders) are fit exclusively on `X_train` to prevent data leakage.